In [4]:
import numpy as np
import xarray as xr
import pandas as pd
import logging
import os
import sys
import matplotlib.pyplot as plt
import matplotlib as mpl
import matplotlib.colors as mcolors
from matplotlib.colors import LinearSegmentedColormap

import xarray as xr
import xesmf as xe
import numpy as np

In [5]:
def regrid_modis_to_wrf_nearest_timeseries(modis_data, nan=False, reuse_weights=False):
    """
    Regrid MODIS 0/1 snow mask to WRF (curvilinear) grid using nearest-neighbor (2D),
    for multiple time steps.

    Parameters
    ----------
    modis_data : xr.DataArray
        Must include a time dimension (any name) plus 2D spatial dims.
        Must have coords 'lat' and 'lon' (1D or 2D).
        Values: 0/1 (optionally NaN for clouds).
    reuse_weights : bool
        If True, reuse saved weights file.

    Returns
    -------
    xr.Dataset with:
      - LAI(time, south_north, west_east) uint8
      - lat(south_north, west_east), lon(south_north, west_east)
    """
    import xesmf as xe
    import xarray as xr
    import numpy as np

    # -----------------------
    # Load WRF template
    # -----------------------
    path_geog_file = "/bsuscratch/stanleyakor/uppercolorado/static_inputs/wrfout_d02_2000-04-08_00:00:00"
    grid_template = xr.open_dataset(path_geog_file)
    LAT = grid_template["XLAT"].isel(Time=0)
    LON = grid_template["XLONG"].isel(Time=0)

    # Target grid 
    target_grid = xr.Dataset(
        {"lat": (LAT.dims, LAT.values),
         "lon": (LON.dims, LON.values)}
    )

    
    if "lat" not in modis_data.coords or "lon" not in modis_data.coords:
        raise ValueError("modis_data must have 'lat' and 'lon' coordinates (1D or 2D).")

    #
    candidate_time = [d for d in modis_data.dims if d.lower() in ("time", "xtime", "t")]
    if candidate_time:
        tdim = candidate_time[0]
    else:
        # fallback: first dim that is not lat/lon dims (common: time,y,x)
        tdim = None
        for d in modis_data.dims:
            if d not in ("lat", "lon"):
                tdim = d
                break

   
    spatial_dims = [d for d in modis_data.dims if d != tdim]
    if len(spatial_dims) != 2:
        raise ValueError(
            f"Expected 2 spatial dims (plus optional time), got dims={modis_data.dims}. "
            "Make sure modis_data is (time, y, x) or (y, x)."
        )
    ydim, xdim = spatial_dims

    
    if modis_data["lat"].ndim == 1:
        
        
        if float(modis_data["lat"].diff("lat").mean()) < 0:
            modis_data = modis_data.sortby("lat")
    else:
        
        lat_profile = modis_data["lat"].isel({xdim: 0})
        if float(lat_profile.diff(ydim).mean()) < 0:
            modis_data = modis_data.isel({ydim: slice(None, None, -1)})

    # -----------------------
    # Build explicit MODIS source grid (2D)
    # -----------------------
    if modis_data["lat"].ndim == 1 and modis_data["lon"].ndim == 1:
        lon2, lat2 = xr.broadcast(modis_data["lon"], modis_data["lat"])
        source_grid = xr.Dataset({"lat": lat2, "lon": lon2})
    else:
        source_grid = xr.Dataset({"lat": modis_data["lat"], "lon": modis_data["lon"]})

    # -----------------------
    # Build regridder once, apply to all timesteps
    # -----------------------
    regridder = xe.Regridder(
        source_grid,
        target_grid,
        method="nearest_s2d",
        periodic=False,
        ignore_degenerate=True,
        filename="modis_to_wrf_weights_nearest_s2d.nc",
        reuse_weights=reuse_weights,
    )


    modis_nn = regridder(modis_data)
 
    modis_nn.name = "LAI"
    out_coords = {"lat": (LAT.dims, LAT.values), "lon": (LON.dims, LON.values)}
    if tdim is not None and tdim in modis_data.coords:
        out_coords[tdim] = modis_data[tdim]

    ds_out = xr.Dataset(
        data_vars={"LAI": modis_nn},
        coords=out_coords,
        attrs={
            "description": "MODIS snow mask regridded to WRF grid using xESMF nearest_s2d (timeseries)",
            "regridding_method": "nearest_s2d",
            "snow_threshold": 0.5,
            "wrf_template": path_geog_file,
        },
    )

    ds_out["LAI"].attrs.update(
        {"long_name": "Leaf area index on WRF grid"}
    )

    return ds_out


In [23]:
# regrid lai data

# lai_data = xr.open_dataset('../data/lai_unregrided.nc')['LAI']

In [24]:
# regridded_lai = regrid_modis_to_wrf_nearest_timeseries(lai_data)

In [25]:
# regridded_lai['LAI'].isel(XTIME = 497).plot()

In [22]:
# regridded_lai.to_netcdf('../data/lai_regrided.nc')